# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Pairwise graphs cannot represent group relationships without introducing distorted clique expansions. Zhou et al. (2006) and Feng et al. (2019) formulated **Hypergraph Neural Networks (HGNN)**, defining spectral message passing on bipartite node-hyperedge incidence matrices. Chien et al. (2022) established **AllSet**, proving that hypergraph convolutions are universal compositions of two multiset functions. Shirzad et al. (2023) developed **Exphormer**, constructing sparse linear $O(V + E)$ Graph Transformers using Ramanujan expander graphs.

### Purpose & Target Architecture
This notebook constructs from first principles:
1. Hypergraph bipartite incidence convolution with degree normalization.
2. AllSet two-phase universal multiset aggregation.
3. Exphormer expander graph spectral message mixing.

# 2. Mathematical Foundations & Governing Equations

### Hypergraph Incidence Matrix
For $V$ vertices and $E$ hyperedges:
$$H_{ve} = 1 \iff v \in e, \quad D_v = \text{diag}\left(\sum_e H_{ve}\right), \quad D_e = \text{diag}\left(\sum_v H_{ve}\right)$$
Hypergraph message passing factorizes into two stages:
$$Z_e = D_e^{-1} H^T X \quad (\text{Node} \to \text{Hyperedge}), \quad X_{\text{agg}} = D_v^{-1} H Z_e \quad (\text{Hyperedge} \to \text{Node})$$

### AllSet Universal Multiset Aggregation
$$Z_e = \text{MLP}_E\left(\sum_{v \in e} \text{MLP}_V(X_v)\right), \quad X_v^{(l+1)} = X_v^{(l)} + \text{MLP}_V^{\text{out}}\left(\sum_{e \ni v} Z_e\right)$$

### Exphormer Sparse Expander Mixing
Using $d$-regular expander graphs with bounded spectral gap:
$$H_{\text{exp}} = A_{\text{exp}} H, \quad H^{(l+1)} = \text{GELU}\left((H + H_{\text{exp}}) W_{\text{mix}} + b_{\text{mix}}\right)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Hypergraph: 4 nodes, 3 hyperedges (e0: {0,1,2}, e1: {1,2,3}, e2: {0,3})
V, E, D = 4, 3, 8
inc = torch.tensor([
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
], dtype=torch.float32)

x = torch.randn(V, D)

# Expander graph (2-regular cycle on 4 nodes)
exp_adj = torch.tensor([
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 0.0],
], dtype=torch.float32)

print(f"Setup complete: V={V} nodes, E={E} hyperedges, feature dimension D={D}")
print(f"Node degrees: {inc.sum(dim=1).tolist()}, Edge degrees: {inc.sum(dim=0).tolist()}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def hypergraph_conv(x: torch.Tensor, inc: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    d_v = torch.sum(inc, dim=1, keepdim=True).clamp(min=1e-6)
    d_e = torch.sum(inc, dim=0, keepdim=True).clamp(min=1e-6)
    z_e = (inc.T @ x) / d_e.T
    x_agg = (inc @ z_e) / d_v
    return F.gelu(x_agg @ w + b)

def allset_multiset_step(x: torch.Tensor, inc: torch.Tensor, w_v: torch.Tensor, b_v: torch.Tensor, w_e: torch.Tensor, b_e: torch.Tensor) -> torch.Tensor:
    h_v = F.relu(x @ w_v + b_v)
    z_e = F.relu((inc.T @ h_v) @ w_e + b_e)
    return x + F.relu((inc @ z_e) @ w_v + b_v)

def exphormer_spectral_mixing(h: torch.Tensor, expander_adj: torch.Tensor, w_mix: torch.Tensor, b_mix: torch.Tensor) -> torch.Tensor:
    h_exp = expander_adj @ h
    return F.gelu((h + h_exp) @ w_mix + b_mix)

w_hg = torch.randn(D, D) * 0.2
b_hg = torch.randn(D) * 0.1
h_hg = hypergraph_conv(x, inc, w_hg, b_hg)

w_v = torch.randn(D, D) * 0.2
b_v = torch.randn(D) * 0.1
w_e = torch.randn(D, D) * 0.2
b_e = torch.randn(D) * 0.1
h_allset = allset_multiset_step(x, inc, w_v, b_v, w_e, b_e)

w_exp = torch.randn(D, D) * 0.2
b_exp = torch.randn(D) * 0.1
h_exphormer = exphormer_spectral_mixing(x, exp_adj, w_exp, b_exp)

print(f"Hypergraph conv output shape: {h_hg.shape}")
print(f"AllSet output shape: {h_allset.shape}")
print(f"Exphormer output shape: {h_exphormer.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: Degree normalization conservation
# If all input node features are constant vector 1, normalized aggregation must preserve 1
ones_x = torch.ones(V, D)
d_v = torch.sum(inc, dim=1, keepdim=True)
d_e = torch.sum(inc, dim=0, keepdim=True)
z_ones = (inc.T @ ones_x) / d_e.T
x_ones_agg = (inc @ z_ones) / d_v
assert torch.allclose(x_ones_agg, ones_x, atol=1e-6), "Degree normalized hypergraph aggregation must preserve constant fields"

# Invariant 2: Permutation equivariance of AllSet
perm = torch.tensor([2, 0, 3, 1])
x_perm = x[perm]
inc_perm = inc[perm]
h_allset_perm = allset_multiset_step(x_perm, inc_perm, w_v, b_v, w_e, b_e)
expected_perm = h_allset[perm]
diff = torch.max(torch.abs(h_allset_perm - expected_perm)).item()
assert diff < 1e-6, f"AllSet must be permutation equivariant: error {diff:.2e}"

# Invariant 3: Expander graph degree regularity (2-regular cycle)
exp_deg = exp_adj.sum(dim=-1)
assert torch.all(exp_deg == 2.0), "Expander graph must be strictly 2-regular"
print(f"All hypergraph and expander invariants passed successfully! Permutation error: {diff:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark Hypergraph Conv vs Dense Clique Expansion
# Expanding hyperedge e0 {0, 1, 2} into a pairwise clique adds 3 edges
clique_adj = torch.zeros(V, V)
for e in range(E):
    nodes_in_e = torch.where(inc[:, e] > 0)[0]
    for u in nodes_in_e:
        for v in nodes_in_e:
            if u != v:
                clique_adj[u, v] = 1.0

start = time.perf_counter()
for _ in range(500):
    _ = hypergraph_conv(x, inc, w_hg, b_hg)
hg_time = (time.perf_counter() - start) / 500.0

start = time.perf_counter()
for _ in range(500):
    _ = F.gelu((clique_adj @ x) @ w_hg + b_hg)
clique_time = (time.perf_counter() - start) / 500.0

print(f"Native hypergraph convolution latency: {hg_time*1e6:.1f} µs")
print(f"Clique-expanded graph convolution latency: {clique_time*1e6:.1f} µs")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Dense clique expansion creates false connectivity paths
# Consider 3 independent research papers (hyperedges) overlapping partially
# Clique expansion creates artificial shortest paths between nodes that never co-occurred
print(f"Original hyperedges: {E} hyperedges")
print(f"Clique expansion creates: {int(clique_adj.sum().item()) // 2} edges (Distorts relational topology)")

# Zero degree division without clamping
def unclamped_hypergraph_bad(x: torch.Tensor, inc: torch.Tensor):
    # If a node or hyperedge has zero degree, plain division gives NaN
    d_v = torch.sum(inc, dim=1, keepdim=True)
    d_e = torch.sum(inc, dim=0, keepdim=True)
    return (inc @ ((inc.T @ x) / d_e.T)) / d_v

inc_isolated = inc.clone()
inc_isolated[0, :] = 0.0  # Node 0 is completely disconnected (0 degree)

bad_out = unclamped_hypergraph_bad(x, inc_isolated)
good_out = hypergraph_conv(x, inc_isolated, w_hg, b_hg)

print(f"Unclamped isolated node output has NaNs: {torch.isnan(bad_out).any().item()} (Fatal crash)")
print(f"Clamped hypergraph output has NaNs: {torch.isnan(good_out).any().item()} (Numerically stable)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Insights
1. **Hypergraphs Avoid Clique Distortion**: Operating directly on bipartite incidence matrices $H$ preserves true multi-way correlations without creating spurious pairwise paths.
2. **Universality of AllSet**: Factoring hypergraph learning into two alternating multiset aggregators yields a universal approximator for higher-order hypergraph functions.
3. **Exphormer Expander Sparsity**: Expander graphs provide optimal expander mixing, delivering the global communicative power of dense Transformers in linear $O(V + E)$ time.